# Paper analysis

This notebook reproduces the final **paper-facing analysis** from the held-out
benchmark results. It deliberately separates descriptive summaries from paired
method comparisons so that statements such as “DMP2 improves over DMP most at
low seed fractions” can be traced to explicit rows and figures rather than to
the ancient oral tradition of remembering what a plotting script once said.

The notebook covers:

1. data-quality and method-availability checks;
2. overall real-world and synthetic benchmark summaries;
3. per-dataset and per-family results;
4. DMP vs. DMP2 paired analysis;
5. ALE / IPL / ALE2 / IPL2 comparisons;
6. SWE, SWE-NO, and cavity comparisons;
7. diffusion-strength, seed-fraction, and edge-model regimes;
8. runtime-accuracy trade-offs;
9. selected-hyperparameter summaries;
10. optional regeneration of the real-network structural-statistics table.

Generated outputs are written to `analysis/figures/` and `analysis/tables/`.

In [ ]:
from pathlib import Path
import sys
import subprocess

import pandas as pd
from IPython.display import display

# Work whether Jupyter was launched from the repository root or analysis/.
_here = Path.cwd().resolve()
ROOT = None
for candidate in [_here, *_here.parents]:
    if (candidate / "results" / "real_world_benchmark" / "benchmark_results.csv").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate repository root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.paper_analysis import (
    METHOD_LABELS,
    load_all,
    data_quality_report,
    pooled_method_summary,
    grouped_method_summary,
    setting_level_summary,
    paired_instances,
    paired_summary,
    paired_by,
    selected_hyperparameter_summary,
    generate_all,
)

print("Repository root:", ROOT)

## 1. Load final held-out benchmark results

Only successful held-out rows are used for accuracy summaries. Failed rows are
retained separately for the audit below.

In [ ]:
data = load_all(ROOT)
real = data["real"]
synthetic = data["synthetic"]

print(f"Real-world successful held-out rows: {len(real):,}")
print(f"Synthetic successful held-out rows: {len(synthetic):,}")
print("Real methods:", sorted(real["method"].unique()))
print("Synthetic methods:", sorted(synthetic["method"].unique()))

## 2. Data-quality and method-availability audit

This should be inspected before interpreting any aggregate. In particular,
historical result archives can predate ALE2/IPL2 or contain isolated failed
method evaluations.

In [ ]:
print("Real-world status counts")
display(data_quality_report(data["real_raw"]))

print("Synthetic status counts")
display(data_quality_report(data["synthetic_raw"]))

availability = []
for bench_name, df in [("real", real), ("synthetic", synthetic)]:
    present = set(df["method"].unique())
    for method, label in METHOD_LABELS.items():
        availability.append({
            "benchmark": bench_name,
            "method": method,
            "Method": label,
            "available": method in present,
        })
display(pd.DataFrame(availability))

## 3. Overall held-out accuracy and runtime

The first table pools successful held-out **instance-level** evaluations. The
second first takes a median across test replicates within each experimental
setting and then pools across settings. Keeping both makes the aggregation rule
explicit, because apparently even medians require constitutional law.

In [ ]:
real_overall = pooled_method_summary(real)
synthetic_overall = pooled_method_summary(synthetic)

print("Real-world pooled instance-level medians")
display(real_overall)

print("Synthetic pooled instance-level medians")
display(synthetic_overall)

print("Real-world setting-level medians")
display(setting_level_summary(real))

print("Synthetic setting-level medians")
display(setting_level_summary(synthetic))

## 4. Per-dataset and per-family summaries

In [ ]:
real_by_dataset = grouped_method_summary(real, "dataset")
synthetic_by_family = grouped_method_summary(synthetic, "graph_family")

display(real_by_dataset)
display(synthetic_by_family)

## 5. DMP2 vs. DMP

For all paired analyses below, **positive RMSE improvement means the first method
is better**:

\[
\Delta_{A,B} = \mathrm{RMSE}(B)-\mathrm{RMSE}(A).
\]

Thus positive values for DMP2 vs. DMP mean DMP2 has lower RMSE.

In [ ]:
for bench_name, df, structural_group in [
    ("real", real, "dataset"),
    ("synthetic", synthetic, "graph_family"),
]:
    pair = paired_instances(df, "dmp_est_r2", "dmp_est")
    print(f"\n{bench_name.upper()}: DMP2 vs DMP")
    display(paired_summary(pair))
    display(paired_by(pair, structural_group))
    display(paired_by(pair, "requested_seed_fraction"))
    display(paired_by(pair, "target_secondary_activation"))
    display(paired_by(pair, ["requested_seed_fraction", "target_secondary_activation"]))

## 6. ALE / IPL / ALE2 / IPL2

These comparisons distinguish temporal noisy-OR accumulation (IPL versus ALE)
from the second-order incoming correction (ALE2/IPL2). A comparison is skipped
when the corresponding method is absent from a historical benchmark archive.

In [ ]:
ale_comparisons = [
    ("IPL vs ALE", "modified_ALE", "ALE_heuristic"),
    ("ALE2 vs ALE", "ALE2", "ALE_heuristic"),
    ("IPL2 vs IPL", "modified_ALE2", "modified_ALE"),
    ("IPL2 vs ALE2", "modified_ALE2", "ALE2"),
]

for bench_name, df in [("real", real), ("synthetic", synthetic)]:
    print(f"\n### {bench_name.upper()}")
    for title, a, b in ale_comparisons:
        pair = paired_instances(df, a, b)
        if pair.empty:
            print(f"{title}: not available in this benchmark archive")
            continue
        print(title)
        display(paired_summary(pair))
        display(paired_by(pair, "requested_seed_fraction"))
        display(paired_by(pair, "target_secondary_activation"))
        display(paired_by(pair, "edge_model"))

## 7. SWE, SWE-NO, cavity variants, and NM

In [ ]:
swe_comparisons = [
    ("SWE vs NM", "swe", "Naive"),
    ("SWE-NO vs SWE", "swe_no", "swe"),
    ("SWE-NO vs NM", "swe_no", "Naive"),
    ("Cavity SWE vs SWE", "swe_cavity", "swe"),
    ("Cavity SWE-NO vs SWE-NO", "swe_no_cavity", "swe_no"),
    ("SWE-HIB-C vs NM", "swe_hib_cavity", "Naive"),
    ("SSS vs NM", "sss", "Naive"),
]

for bench_name, df in [("real", real), ("synthetic", synthetic)]:
    print(f"\n### {bench_name.upper()}")
    for title, a, b in swe_comparisons:
        pair = paired_instances(df, a, b)
        if pair.empty:
            continue
        print(title)
        display(paired_summary(pair))

## 8. Regime summaries

These tables support statements about diffusion strength, seed fraction, and
edge-probability model without requiring a separate ad hoc script for every
sentence in the Results section.

In [ ]:
for bench_name, df in [("real", real), ("synthetic", synthetic)]:
    print(f"\n{bench_name.upper()}: by seed fraction")
    display(grouped_method_summary(df, "requested_seed_fraction"))
    print(f"{bench_name.upper()}: by target secondary activation")
    display(grouped_method_summary(df, "target_secondary_activation"))
    print(f"{bench_name.upper()}: by edge model")
    display(grouped_method_summary(df, "edge_model"))

## 9. Selected hyperparameters

In [ ]:
print("Real-world selected hyperparameters")
display(selected_hyperparameter_summary(data["real_selected"]))

print("Synthetic selected hyperparameters")
display(selected_hyperparameter_summary(data["synthetic_selected"]))

## 10. Generate all standard paper tables and figures

This is the main export cell. It creates the numerical tables used above and
the standard pooled-RMSE, runtime-accuracy, DMP2/DMP, ALE/IPL, and SWE regime
figures.

In [ ]:
outputs = generate_all(ROOT)
print("Figures:", outputs["figures_dir"])
print("Tables:", outputs["tables_dir"])
display(outputs["paired_summary"])

## 11. Optional real-network structural-statistics table

The repository already contains `make_network_statistics_table.py`, which
computes node/edge counts, average degree, clustering, triangle counts, and
reciprocity directly from the raw real-world datasets. It is more expensive
than the CSV-only analysis above, so regeneration is optional.

In [ ]:
RUN_NETWORK_STATISTICS = False

if RUN_NETWORK_STATISTICS:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / "make_network_statistics_table.py"),
            "--output", str(ROOT / "analysis" / "tables" / "network_statistics.tex"),
            "--csv", str(ROOT / "analysis" / "tables" / "network_statistics.csv"),
        ],
        check=True,
        cwd=ROOT,
    )

## 12. Export inventory

In [ ]:
figures = sorted((ROOT / "analysis" / "figures").glob("*.png"))
tables = sorted((ROOT / "analysis" / "tables").glob("*.csv"))
print(f"Generated {len(figures)} figures and {len(tables)} CSV tables.")
for p in figures:
    print("FIG ", p.relative_to(ROOT))
for p in tables:
    print("TABLE", p.relative_to(ROOT))